### Import Library

In [13]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import norm
from scipy.optimize import brentq
import plotly.io as pio

pio.renderers.default = 'colab'


### Import Data

In [14]:
ticker = yf.Ticker("SPY")
S = ticker.history(period="1d")['Close'].iloc[-1]
options_dates = ticker.options

chains = []
# Fetching first 6 expirations for stable 3D surface generation
for date in options_dates[:6]:
    opt = ticker.option_chain(date)
    calls = opt.calls
    calls['expirationDate'] = date
    chains.append(calls)

options_chain = pd.concat(chains, ignore_index=True)

### Describe Data

In [15]:
print(f"Spot Price (S): ${S:.2f}")
print(f"Total Raw Contracts: {len(options_chain)}")
display(options_chain.head())
options_chain.info()

Spot Price (S): $743.23
Total Raw Contracts: 654


,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,expirationDate
0,SPY260708C00625000,2026-07-07 14:33:47+00:00,625.0,121.80,113.75,117.11,0.000000,0.000000,30.0,15,0.00001,True,REGULAR,USD,2026-07-08
1,SPY260708C00640000,2026-06-25 19:58:59+00:00,640.0,93.57,98.80,102.13,0.000000,0.000000,NaN,1,0.00001,True,REGULAR,USD,2026-07-08
2,SPY260708C00650000,2026-07-07 14:36:56+00:00,650.0,96.42,88.75,92.11,0.000000,0.000000,1.0,1,0.00001,True,REGULAR,USD,2026-07-08
3,SPY260708C00675000,2026-07-07 14:29:01+00:00,675.0,73.18,63.62,66.42,0.000000,0.000000,1.0,2,0.00001,True,REGULAR,USD,2026-07-08
4,SPY260708C00680000,2026-07-07 13:55:02+00:00,680.0,60.39,60.35,60.54,-8.860001,-12.794225,12.0,11,0.00001,True,REGULAR,USD,2026-07-08


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 654 entries, 0 to 653
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   contractSymbol     654 non-null    object             
 1   lastTradeDate      654 non-null    datetime64[ns, UTC]
 2   strike             654 non-null    float64            
 3   lastPrice          654 non-null    float64            
 4   bid                654 non-null    float64            
 5   ask                654 non-null    float64            
 6   change             654 non-null    float64            
 7   percentChange      654 non-null    float64            
 8   volume             603 non-null    float64            
 9   openInterest       654 non-null    int64              
 10  impliedVolatility  654 non-null    float64            
 11  inTheMoney         654 non-null    bool               
 12  contractSize       654 non-null    object         


### Data Visualization

In [16]:
hist = ticker.history(period="1y")
fig = go.Figure(data=go.Scatter(x=hist.index, y=hist['Close'], mode='lines', name='SPY Spot'))
fig.update_layout(title="Underlying Asset (SPY) - 1Y Historical Trend", xaxis_title="Date", yaxis_title="Price")
fig.show()


### Data Preprocessing

In [17]:
# 1. Liquidity Scrub
options_chain = options_chain[options_chain['volume'] > 0]

# 2. Time to Maturity (T) Calculation - Timezone stripped to prevent Pandas errors
options_chain['expirationDate'] = pd.to_datetime(options_chain['expirationDate']).dt.tz_localize(None)
today = pd.Timestamp.today().tz_localize(None)
options_chain['T'] = (options_chain['expirationDate'] - today).dt.days / 365.25

# Drop expired contracts
options_chain = options_chain[options_chain['T'] > 0]

# 3. Moneyness Standardization
options_chain['Moneyness'] = options_chain['strike'] / S
options_chain = options_chain[(options_chain['Moneyness'] >= 0.85) & (options_chain['Moneyness'] <= 1.15)]

df = options_chain.copy()

# Macro Constants
r = 0.043
q = 0.013

### Define Target Variable (y) and Feature Variables (X)
System Architecture Mapping:
Features (X): Spot Price (S), Strike Price (K), Time to Maturity (T), Risk-Free Rate (r), Dividend Yield (q).
Target (y): Implied Volatility (σ). Extracted iteratively from Market Premium.

### Train Test Split
Split Invalid.
Black-Scholes-Merton functions as a deterministic analytical engine. Probabilistic gradient descent training obsolete. Implied Volatility extracted via real-time numerical root-finding (Brent's Method).

### Modeling

In [18]:
# Execute Extraction
df['IV'] = df.apply(calculate_iv_complete, axis=1)
df.dropna(subset=['IV'], inplace=True)

# Render Graphics
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "xy"}, {"type": "scene"}]], subplot_titles=("Front-Month Volatility Smile", "3D Volatility Surface"))

valid_maturities = df['T'].value_counts()
best_T = valid_maturities[valid_maturities > 4].index.min()
front_month = df[df['T'] == best_T]

fig.add_trace(go.Scatter(x=front_month['Moneyness'], y=front_month['IV'], mode='lines+markers', marker=dict(color=front_month['IV'], colorscale='Viridis', size=8)), row=1, col=1)
fig.add_vline(x=1.0, line_dash="dash", line_color="red", row=1, col=1, annotation_text="Spot (ATM)")

fig.add_trace(go.Mesh3d(x=df['Moneyness'], y=df['T'], z=df['IV'], intensity=df['IV'], colorscale='Viridis', opacity=0.9), row=1, col=2)

fig.update_layout(title=f"Options Analytics Engine | SPY Spot: ${S:.2f}", height=650, margin=dict(l=20, r=20, b=20, t=60), showlegend=False, hovermode="x unified")

# FIXED: Slider preserved, explicit range constraint added to prevent 100k blowout
fig.update_xaxes(title_text="Moneyness (Strike/Spot)", range=[0.85, 1.15], rangeslider=dict(visible=True, thickness=0.1), row=1, col=1)
fig.update_yaxes(title_text="Implied Volatility", row=1, col=1)
fig.layout.scene.update(xaxis_title="Moneyness", yaxis_title="Maturity (Yrs)", zaxis_title="IV")

fig.show(config={'scrollZoom': True, 'displayModeBar': True, 'modeBarButtonsToRemove': ['lasso2d', 'select2d']})

# EXPORT: Saves interactive slider layout directly to deployment file
fig.write_html("volatility_surface.html")